In [0]:
dim_table=dbutils.jobs.taskValues.get(taskKey="Gold_Dimension", key="dim_table", debugValue="onlineretail_ws.gold.dim_customer")
dbutils.widgets.text("fact_path","abfss://gold@onlineretailproject.dfs.core.windows.net/fact_revenue")
fact_path=dbutils.widgets.get("fact_path")
dbutils.widgets.text("fact_revenue_table","onlineretail_ws.gold.fact_revenue")
fact_revenue_table=dbutils.widgets.get("fact_revenue_table")

In [0]:
spark.sql(f"""
          select * from {dim_table}
          """)

In [0]:
revenue_df=spark.sql(f"""
    SELECT DISTINCT
        Customer_ID,
        Country,
        count(Invoice) OVER (PARTITION BY Customer_ID) AS nooforders,
        sum(Price) OVER (PARTITION BY Customer_ID) AS totalrevenue,
        sum(Price) OVER (PARTITION BY Country) AS revenue_by_country
    FROM {dim_table}
    WHERE Customer_ID IS NOT NULL
""")

In [0]:
revenue_df.write.format('delta').option("mergeSchema", "true").option("overwriteSchema", "true").mode("append").option("path",fact_path).saveAsTable(fact_revenue_table)